# Notebook 2 — Monte Carlo Clinical Trial Simulator
### AMPK Signaling Case Study: Metformin at the Triad

Section 4 of the case study write-up notes a puzzle: strong epidemiological signals suggested
metformin lowered cancer risk, but large randomized controlled trials largely failed to confirm
this. This notebook builds a **Monte Carlo simulation** to show *why* that gap can happen even when
a real (but modest) drug effect exists — combining two separate statistical phenomena:

1. **Confounding by indication / healthy-user bias** — in observational data, people who get
   prescribed and stay on metformin differ systematically from those who don't, which can make an
   observational effect estimate look stronger than the true causal effect.
2. **Underpowered trials** — randomized trials are unbiased, but if the true effect is modest and
   the trial is too small, it may simply fail to reach statistical significance by chance, even with
   a real effect present.

We simulate thousands of virtual trials under each scenario and compare what each design "sees."


In [ ]:
!pip install ipywidgets -q
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider, Checkbox


### Core trial simulator

In [ ]:
def simulate_trial(n_per_arm, baseline_risk, true_rr, randomized=True,
                    confounding_bias=0.0, rng=None):
    """
    Simulate one virtual two-arm trial (metformin vs control) for a binary outcome
    (e.g. cancer incidence over the follow-up period).

    true_rr        : true relative risk for the treated arm vs control (e.g. 0.90 = 10% true reduction)
    randomized     : if False, applies `confounding_bias` to mimic healthy-user bias in
                      observational (non-randomized) data
    confounding_bias : extra apparent risk reduction from bias, applied only when randomized=False
    """
    if rng is None:
        rng = np.random.default_rng()

    control_risk = baseline_risk
    if randomized:
        treated_risk = baseline_risk * true_rr
    else:
        treated_risk = baseline_risk * true_rr * (1 - confounding_bias)

    treated_events = rng.binomial(n_per_arm, np.clip(treated_risk, 0, 1))
    control_events = rng.binomial(n_per_arm, np.clip(control_risk, 0, 1))

    table = [[treated_events, n_per_arm - treated_events],
             [control_events, n_per_arm - control_events]]
    chi2, p, _, _ = stats.chi2_contingency(table, correction=False)

    p_treated = treated_events / n_per_arm
    p_control = control_events / n_per_arm
    est_rr = (p_treated / p_control) if p_control > 0 else np.nan

    return p, est_rr


def monte_carlo(n_per_arm, baseline_risk, true_rr, randomized, confounding_bias,
                 n_trials=2000, alpha=0.05, seed=1):
    rng = np.random.default_rng(seed)
    ps, rrs = [], []
    for _ in range(n_trials):
        p, est_rr = simulate_trial(n_per_arm, baseline_risk, true_rr, randomized,
                                    confounding_bias, rng=rng)
        ps.append(p); rrs.append(est_rr)
    ps, rrs = np.array(ps), np.array(rrs)
    power = np.mean(ps < alpha)
    return power, rrs, ps


### Reproducing the observed pattern

We give both study types the *same* true effect (`true_rr = 0.90`, a genuine 10% risk reduction),
but let the observational study carry realistic healthy-user bias and a much larger sample size
(registries are big), while the RCT is unbiased but smaller (RCTs are expensive and slower to
enroll).

In [ ]:
power_obs, rrs_obs, ps_obs = monte_carlo(
    n_per_arm=3000, baseline_risk=0.08, true_rr=0.90,
    randomized=False, confounding_bias=0.15, n_trials=1500, seed=10)

power_rct, rrs_rct, ps_rct = monte_carlo(
    n_per_arm=600, baseline_risk=0.08, true_rr=0.90,
    randomized=True, confounding_bias=0.0, n_trials=1500, seed=11)

print(f"Observational studies : power = {power_obs:.0%}   median apparent RR = {np.nanmedian(rrs_obs):.2f}")
print(f"Randomized trials     : power = {power_rct:.0%}   median apparent RR = {np.nanmedian(rrs_rct):.2f}")
print(f"(true RR = 0.90 in both scenarios)")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12,4.5))

axes[0].hist(rrs_obs, bins=30, alpha=0.6, color='#6B3F6E', label='Observational (biased)')
axes[0].hist(rrs_rct, bins=30, alpha=0.6, color='#1F6F6B', label='RCT (unbiased)')
axes[0].axvline(0.90, color='k', linestyle='--', label='True RR = 0.90')
axes[0].set_xlabel('Estimated relative risk'); axes[0].set_ylabel('Number of virtual trials')
axes[0].set_title('Effect estimates: biased vs unbiased design'); axes[0].legend()

axes[1].bar(['Observational\n(n=3000/arm)', 'RCT\n(n=600/arm)'], [power_obs, power_rct],
            color=['#6B3F6E', '#1F6F6B'])
axes[1].axhline(0.8, color='k', linestyle=':', label='80% power convention')
axes[1].set_ylabel('Statistical power (fraction reaching p<0.05)')
axes[1].set_title('Why one design "sees" the effect and the other often doesn\'t')
axes[1].legend(); axes[1].set_ylim(0,1)

plt.tight_layout(); plt.show()


**Reading this plot:** the observational distribution is both *shifted* (biased toward an
apparently stronger effect) and *narrow* (large N → precise estimates, so trials reliably hit
significance). The RCT distribution is *centered correctly* on the true effect but *wide* (smaller
N → noisy estimates), so many individual RCTs land on the "no significant difference" side of
p = 0.05 purely by chance, even though the true effect is real.

### Power curve: how much would an RCT need to grow to reliably detect a true 10% risk reduction?

In [ ]:
sample_sizes = [100, 250, 500, 1000, 2000, 4000, 8000]
powers = []
for n in sample_sizes:
    pw, _, _ = monte_carlo(n_per_arm=n, baseline_risk=0.08, true_rr=0.90,
                            randomized=True, confounding_bias=0.0, n_trials=800, seed=5)
    powers.append(pw)

plt.figure(figsize=(7,4.5))
plt.plot(sample_sizes, powers, marker='o', color='#1F6F6B')
plt.axhline(0.8, color='k', linestyle=':', label='80% power convention')
plt.xscale('log')
plt.xlabel('Patients per arm (log scale)'); plt.ylabel('Statistical power')
plt.title('RCT sample size needed to reliably detect a true 10% risk reduction')
plt.legend(); plt.tight_layout(); plt.show()


### Interactive explorer
Vary the true effect size, bias strength, and sample sizes to see how the two study designs diverge.

In [ ]:
@interact(
    true_rr=FloatSlider(min=0.5, max=1.0, step=0.02, value=0.90, description='True RR'),
    confounding_bias=FloatSlider(min=0.0, max=0.4, step=0.02, value=0.15, description='Obs. bias'),
    n_obs=IntSlider(min=200, max=6000, step=200, value=3000, description='N obs/arm'),
    n_rct=IntSlider(min=100, max=3000, step=100, value=600, description='N rct/arm'),
)
def explore(true_rr=0.90, confounding_bias=0.15, n_obs=3000, n_rct=600):
    power_obs, rrs_obs, _ = monte_carlo(n_obs, 0.08, true_rr, False, confounding_bias, n_trials=800, seed=20)
    power_rct, rrs_rct, _ = monte_carlo(n_rct, 0.08, true_rr, True, 0.0, n_trials=800, seed=21)

    fig, ax = plt.subplots(figsize=(7,4.5))
    ax.hist(rrs_obs, bins=25, alpha=0.6, color='#6B3F6E', label=f'Observational (power={power_obs:.0%})')
    ax.hist(rrs_rct, bins=25, alpha=0.6, color='#1F6F6B', label=f'RCT (power={power_rct:.0%})')
    ax.axvline(true_rr, color='k', linestyle='--', label=f'True RR={true_rr:.2f}')
    ax.legend(); ax.set_xlabel('Estimated relative risk')
    plt.tight_layout(); plt.show()


### Discussion prompts for your write-up

- At what combination of bias strength and RCT sample size does the RCT "catch up" to the
  observational study's power? What does that imply about trial design trade-offs?
- The placental context (Section 5) also relies mostly on small trials so far. If you plug in
  parameters resembling a small preeclampsia trial (e.g. `n_rct` around 50–150 per arm), what does
  that suggest about the risk of a real placental benefit going undetected?
- This model assumes bias only shifts the mean effect. In reality, bias can also inflate variance
  (e.g. through unmeasured confounders). How might you extend `simulate_trial` to capture that?
